# 🚗 Traffic Demerit System - ML Training (Google Colab)

This notebook trains the complete ML stack for the **Traffic Demerit System**:

1. **Synthetic dataset generation** - 2,500 records with 6 non-demographic features
2. **EDA** - distributions, outliers, correlation heatmap, class balance
3. **k-Means segmentation** - elbow plot, silhouette, cluster labels, PCA coords
4. **Association rule mining** - FP-Growth with lecture example verification
5. **Supervised risk model** - ablation study, calibrated RandomForest, fairness report

### Data Provenance
All data is **synthetic** (seed 42). It demonstrates the pipeline and says nothing about real-world accuracy. Demographic attributes (gender, region, occupation, age) are **excluded** from every model.

All artifacts are saved to `/mnt/data/traffic_demerit_ml/` and can be downloaded.

## Step 1: Install Dependencies

Install packages with versions matching the project `requirements.txt`.

In [ ]:
!pip install -q fastapi==0.116.1 uvicorn==0.35.0 pandas==2.3.2 scikit-learn==1.7.1 joblib==1.5.2 numpy==2.3.2
!pip install -q mlxtend==0.22.2 kneed==0.8.6 shap==0.44.0 pymysql==1.1.0 python-dotenv==1.0.1 matplotlib==3.8.0 seaborn==0.13.0

: 

In [ ]:
from __future__ import annotations

import json
import os
import warnings
from pathlib import Path
from collections import Counter

import joblib
import numpy as np
import pandas as pd
import matplotlib
matplotlib.use("Agg")
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display, Image, FileLink

from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier
from sklearn.cluster import KMeans
from sklearn.decomposition import PCA
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.calibration import CalibratedClassifierCV
from sklearn.metrics import (
    roc_auc_score, f1_score, precision_score, recall_score,
    classification_report, silhouette_score
)
from sklearn.model_selection import StratifiedKFold, train_test_split
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from kneed import KneeLocator
from mlxtend.frequent_patterns import apriori, fpgrowth
from mlxtend.frequent_patterns import association_rules as mlxtend_rules

warnings.filterwarnings("ignore")
np.random.seed(42)

# Output directories
OUTPUT_DIR = Path("/mnt/data/traffic_demerit_ml")
MODEL_DIR = OUTPUT_DIR / "model"
REPORTS_DIR = OUTPUT_DIR / "reports"
EDA_DIR = REPORTS_DIR / "eda"
DATA_DIR = OUTPUT_DIR / "data"

for d in [OUTPUT_DIR, MODEL_DIR, REPORTS_DIR, EDA_DIR, DATA_DIR]:
    d.mkdir(parents=True, exist_ok=True)

print(f"Output directory: {OUTPUT_DIR}")
print(f"Model directory: {MODEL_DIR}")
print(f"Reports directory: {REPORTS_DIR}")

: 

## Step 2: Generate Synthetic Dataset

Creates 2,500 records (seed 42) with six features:
- `days_since_last_offense`: 0-365
- `speed_recorded`: normal(78, 20) clipped to 0-180
- `zone_type`: urban/highway/school_zone/residential
- `weather_conditions`: clear/rain/fog/night
- `cumulative_active_demerit_points`: 0-30
- `past_fine_payment_delays_days`: gamma(2, 3) clipped to 0-45
- `reoffended_within_180_days`: binary target from logistic formula

No demographic attributes are included.

In [ ]:
rng = np.random.default_rng(42)
n = 2500

zone_types = np.array(["urban", "highway", "school_zone", "residential"])
weather_types = np.array(["clear", "rain", "fog", "night"])

days_since_last_offense = rng.integers(0, 366, size=n)
speed_recorded = rng.normal(78, 20, size=n).clip(0, 180)
zone_type = rng.choice(zone_types, size=n, p=[0.45, 0.25, 0.15, 0.15])
weather_conditions = rng.choice(weather_types, size=n, p=[0.55, 0.2, 0.15, 0.1])
cumulative_active_demerit_points = rng.integers(0, 31, size=n)
past_fine_payment_delays_days = rng.gamma(shape=2.0, scale=3.0, size=n).clip(0, 45)

zone_risk = {"urban": 0.08, "highway": 0.12, "school_zone": 0.18, "residential": 0.06}
weather_risk = {"clear": 0.02, "rain": 0.08, "fog": 0.13, "night": 0.10}

logits = (
    1.3 * (1 - np.minimum(days_since_last_offense, 365) / 365)
    + 1.5 * (cumulative_active_demerit_points / 30)
    + 1.1 * np.minimum(past_fine_payment_delays_days, 45) / 45
    + 0.9 * np.maximum(speed_recorded - 80, 0) / 70
    + np.vectorize(zone_risk.get)(zone_type)
    + np.vectorize(weather_risk.get)(weather_conditions)
    - 2.0
)

probs = 1 / (1 + np.exp(-logits))
reoffended = rng.binomial(1, probs)

df = pd.DataFrame({
    "days_since_last_offense": days_since_last_offense,
    "speed_recorded": speed_recorded.round(2),
    "zone_type": zone_type,
    "weather_conditions": weather_conditions,
    "cumulative_active_demerit_points": cumulative_active_demerit_points,
    "past_fine_payment_delays_days": past_fine_payment_delays_days.round(2),
    "reoffended_within_180_days": reoffended,
})

csv_path = DATA_DIR / "simulated_reoffending.csv"
df.to_csv(csv_path, index=False)

print(f"Dataset saved to {csv_path}")
print(f"Total rows: {len(df)}")
print(f"Class balance: {df['reoffended_within_180_days'].value_counts().to_dict()}")
print(f"Class 1 %: {df['reoffended_within_180_days'].mean() * 100:.1f}%")
display(df.head(10))

## Step 3: Exploratory Data Analysis (EDA)

Generates missing values report, distribution histograms, outlier boxplots, correlation heatmap, and class balance summary.

In [ ]:
FEATURE_COLUMNS = [
    "days_since_last_offense",
    "speed_recorded",
    "zone_type",
    "weather_conditions",
    "cumulative_active_demerit_points",
    "past_fine_payment_delays_days",
]
TARGET_COLUMN = "reoffended_within_180_days"
NUMERIC_COLS = [
    "days_since_last_offense",
    "speed_recorded",
    "cumulative_active_demerit_points",
    "past_fine_payment_delays_days",
]

# --- Missing values report ---
missing = df.isnull().sum()
missing_pct = (df.isnull().mean() * 100).round(2)
missing_report = {
    col: {"missing_count": int(missing[col]), "missing_percent": float(missing_pct[col])}
    for col in df.columns
}
with open(REPORTS_DIR / "missing_values_report.json", "w") as f:
    json.dump(missing_report, f, indent=2)
print("Missing values report:")
display(pd.DataFrame(missing_report).T)

# --- Distribution histograms ---
for col in NUMERIC_COLS:
    plt.figure(figsize=(8, 5))
    sns.histplot(df[col].dropna(), bins=30, kde=True)
    plt.title(f"Distribution of {col}")
    plt.xlabel(col)
    plt.ylabel("Count")
    plt.tight_layout()
    plt.savefig(EDA_DIR / f"dist_{col}.png", dpi=100)
    plt.close()
print(f"Distribution plots: {len(NUMERIC_COLS)} PNGs saved")

# --- Outlier boxplots ---
for col in NUMERIC_COLS:
    plt.figure(figsize=(8, 4))
    sns.boxplot(x=df[col].dropna())
    plt.title(f"Boxplot: {col} (outliers)")
    plt.xlabel(col)
    plt.tight_layout()
    plt.savefig(EDA_DIR / f"outliers_{col}.png", dpi=100)
    plt.close()
print(f"Outlier plots: {len(NUMERIC_COLS)} PNGs saved")

# --- Correlation heatmap ---
corr_cols = [c for c in NUMERIC_COLS if c in df.columns] + [TARGET_COLUMN]
corr = df[corr_cols].corr()
plt.figure(figsize=(10, 8))
sns.heatmap(corr, annot=True, fmt=".2f", cmap="coolwarm", center=0)
plt.title("Feature Correlation Heatmap")
plt.tight_layout()
plt.savefig(EDA_DIR / "correlation_heatmap.png", dpi=100)
plt.close()
print("Correlation heatmap saved")

# --- Class balance ---
counts = df[TARGET_COLUMN].value_counts().to_dict()
total = len(df)
class_balance = {
    "total": int(total),
    "class_0_count": int(counts.get(0, 0)),
    "class_1_count": int(counts.get(1, 0)),
    "class_0_pct": round(float(counts.get(0, 0)) / total * 100, 2),
    "class_1_pct": round(float(counts.get(1, 0)) / total * 100, 2),
}
with open(REPORTS_DIR / "class_balance.json", "w") as f:
    json.dump(class_balance, f, indent=2)
print(f"Class balance: {class_balance}")

# --- EDA summary markdown ---
lines = []
lines.append("# EDA Summary")
lines.append("")
lines.append(f"Total driver records: {len(df)}")
lines.append("All records are synthetic (seed 42).")
lines.append("")
lines.append("## Class Balance (reoffended_within_180_days)")
for k, v in counts.items():
    lines.append(f"- Class {k}: {v} ({v/len(df)*100:.1f}%)")
lines.append("")
lines.append("## Numeric Feature Summary")
lines.append(df[NUMERIC_COLS].describe().to_markdown())
lines.append("")
lines.append("## Data Limitations")
lines.append("- All data is synthetic (generated locally, seed 42).")
lines.append("- Demographic attributes (gender, region, occupation, age) are excluded.")
lines.append("- Results demonstrate the pipeline and say nothing about real-world accuracy.")
with open(REPORTS_DIR / "eda_summary.md", "w") as f:
    f.write("\n".join(lines))
print("EDA summary markdown saved")

display(Image(filename=str(EDA_DIR / "correlation_heatmap.png")))

## Step 4: k-Means Driver Segmentation

Builds a per-driver feature table, then:
- Standardizes features with `StandardScaler`
- Computes WCSS for k=1..10 + elbow plot with kneedle
- Computes silhouette scores for k=2..10
- Selects k (silhouette max near elbow if they disagree)
- Trains final KMeans, auto-labels segments, assigns every driver
- Computes PCA coordinates for scatter plot

Features: violation_count, cumulative_active_demerit_points, avg_speed_over_limit,
past_fine_payment_delays_days, unpaid_fine_ratio, days_since_last_offense, share_at_night

In [ ]:
# Build per-driver feature table (mimics ml_driver_features ETL output)
driver_features_rows = []
for did in range(1, 401):
    violation_count = int(rng.integers(1, 7))
    cumulative_points = int(rng.integers(0, 40))
    days_since_last = int(rng.integers(0, 366))
    new_driver = violation_count < 2
    
    speed_over = np.maximum(rng.normal(15, 10, size=violation_count), 0)
    avg_speed_over_limit = float(speed_over.mean())
    
    share_school = float(rng.beta(2, 8))
    share_night = float(rng.beta(2, 8))
    total_fines = float(rng.integers(0, 50000))
    avg_delay = float(rng.integers(0, 90))
    unpaid_ratio = float(rng.beta(2, 5))
    appeals_filed = int(rng.integers(0, 4))
    
    reoffended = int(rng.random() < (0.15 + 0.55 * (cumulative_points / 30) + 0.2 * violation_count / 10))
    reoffended = min(reoffended, 1)
    
    driver_features_rows.append({
        "driver_profile_id": did,
        "violation_count": violation_count,
        "cumulative_active_demerit_points": cumulative_points,
        "days_since_last_offense": days_since_last,
        "avg_speed_over_limit": round(avg_speed_over_limit, 2),
        "share_in_school_zone": round(share_school, 4),
        "share_at_night": round(share_night, 4),
        "total_fines_kes": round(total_fines, 2),
        "past_fine_payment_delays_days": round(avg_delay, 2),
        "unpaid_fine_ratio": round(unpaid_ratio, 4),
        "appeals_filed": appeals_filed,
        "reoffended_within_180_days": reoffended,
        "new_driver": int(new_driver),
        "is_synthetic": 1,
    })

feature_df = pd.DataFrame(driver_features_rows)
print(f"Driver features: {len(feature_df)} rows")
display(feature_df.head(5))

# --- Clustering ---
CLUSTERING_FEATURES = [
    "violation_count",
    "cumulative_active_demerit_points",
    "avg_speed_over_limit",
    "past_fine_payment_delays_days",
    "unpaid_fine_ratio",
    "days_since_last_offense",
    "share_at_night",
]

RANDOM_STATE = 42
N_INIT = 25

X = feature_df[CLUSTERING_FEATURES].copy()
X = X.fillna(X.median(numeric_only=True))

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# PCA for scatter plot
pca_coords = PCA(n_components=2, random_state=RANDOM_STATE).fit_transform(X_scaled)

# WCSS for k=1..10
wcss = {}
silhouettes = {}
models = {}

for k in range(1, 11):
    km = KMeans(n_clusters=k, n_init=N_INIT, random_state=RANDOM_STATE, max_iter=300)
    km.fit(X_scaled)
    wcss[k] = km.inertia_
    models[k] = km
    if k >= 2:
        sil = silhouette_score(X_scaled, km.labels_)
        silhouettes[k] = sil

print("WCSS and Silhouette by k:")
for k in range(1, 11):
    if k in silhouettes:
        print(f"  k={k}: WCSS={wcss[k]:.2f}, Silhouette={silhouettes[k]:.4f}")
    else:
        print(f"  k={k}: WCSS={wcss[k]:.2f}")

# Kneedle elbow
kl = KneeLocator(list(wcss.keys()), list(wcss.values()), curve="convex", direction="decreasing")
elbow_k = kl.elbow if kl.elbow else 3
print(f"\nKneedle elbow: k={elbow_k}")

if silhouettes:
    best_sil_k = max(silhouettes, key=silhouettes.get)
    print(f"Silhouette best: k={best_sil_k} (score={silhouettes[best_sil_k]:.4f})")

if best_sil_k != elbow_k and silhouettes:
    candidates = [k for k in silhouettes if abs(k - elbow_k) <= 1]
    if candidates:
        chosen_k = max(candidates, key=lambda k: silhouettes[k])
        print(f"Elbow ({elbow_k}) and silhouette ({best_sil_k}) disagree.")
        print(f"Choosing silhouette maximum near elbow: k={chosen_k}")
    else:
        chosen_k = elbow_k
else:
    chosen_k = elbow_k

print(f"\nSelected k = {chosen_k}")

# Elbow plot
plt.figure(figsize=(8, 5))
ks = list(wcss.keys())
wcss_vals = [wcss[k] for k in ks]
plt.plot(ks, wcss_vals, "bo-", linewidth=2, markersize=8)
plt.axvline(x=chosen_k, color="r", linestyle="--", label=f"Chosen k={chosen_k}")
plt.xlabel("k")
plt.ylabel("WCSS (Inertia)")
plt.title("Elbow Plot: WCSS vs k")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(EDA_DIR / "elbow_plot.png", dpi=100)
plt.close()

# Silhouette plot
plt.figure(figsize=(8, 5))
sil_ks = list(silhouettes.keys())
sil_vals = [silhouettes[k] for k in sil_ks]
plt.plot(sil_ks, sil_vals, "go-", linewidth=2, markersize=8)
plt.axvline(x=chosen_k, color="r", linestyle="--", label=f"Chosen k={chosen_k}")
plt.xlabel("k")
plt.ylabel("Silhouette Score")
plt.title("Silhouette Score vs k")
plt.legend()
plt.grid(True, alpha=0.3)
plt.tight_layout()
plt.savefig(EDA_DIR / "silhouette_plot.png", dpi=100)
plt.close()

# Final KMeans
final_km = KMeans(n_clusters=chosen_k, n_init=N_INIT, random_state=RANDOM_STATE, max_iter=300)
final_km.fit(X_scaled)
labels = final_km.labels_
final_silhouette = silhouette_score(X_scaled, labels) if chosen_k >= 2 else None
print(f"Final WCSS: {final_km.inertia_:.2f}")
print(f"Final Silhouette: {final_silhouette:.4f}" if final_silhouette else "N/A (k=1)")

# Centroids in original units
centroids_scaled = final_km.cluster_centers_
centroids_original = scaler.inverse_transform(centroids_scaled)
centroids_df = pd.DataFrame(centroids_original, columns=CLUSTERING_FEATURES)

# Auto-label segments
def label_segment(row):
    vc = row["violation_count"]
    points = row["cumulative_active_demerit_points"]
    speed = row["avg_speed_over_limit"]
    delays = row["past_fine_payment_delays_days"]
    unpaid = row["unpaid_fine_ratio"]
    nights = row["share_at_night"]
    recency = row["days_since_last_offense"]
    
    if vc < 2:
        return "New / insufficient history"
    if points > 20 and unpaid > 0.5:
        return "Chronic late payers"
    if speed > 20 and nights > 0.4:
        return "Habitual night-time speeders"
    if speed > 15:
        return "Habitual speeders"
    if nights > 0.5:
        return "Night-time offenders"
    if points > 10:
        return "High-risk repeat offenders"
    if vc >= 2 and points <= 5 and recency > 90:
        return "Low-risk compliant"
    return f"Segment {int(row.name)}"

labels_generated = [label_segment(r) for _, r in centroids_df.iterrows()]
label_counts = Counter(labels_generated)
seen = {}
unique_labels = []
for lbl in labels_generated:
    if label_counts[lbl] > 1:
        seen[lbl] = seen.get(lbl, 0) + 1
        unique_labels.append(f"{lbl} #{seen[lbl]}")
    else:
        unique_labels.append(lbl)
centroids_df["segment_label"] = unique_labels
print("\nSegment labels:")
for i, row in centroids_df.iterrows():
    print(f"  Cluster {i}: {row['segment_label']}")

# Distances to centroid
distances = final_km.transform(X_scaled)
nearest_centroid = distances.argmin(axis=1)
min_distance = distances.min(axis=1)
dist_mean = min_distance.mean()
dist_std = min_distance.std()
atypical_threshold = dist_mean + 2 * dist_std
is_atypical = min_distance > atypical_threshold
print(f"\nAtypical threshold: distance > {atypical_threshold:.4f}")
print(f"Atypical drivers: {int(is_atypical.sum())}")

# Save model artifacts
joblib.dump(scaler, MODEL_DIR / "kmeans_scaler.joblib")
joblib.dump(final_km, MODEL_DIR / "kmeans_model.joblib")

# Save segment_models metadata as JSON (for the DB upsert later)
elbow_data = {str(k): float(wcss[k]) for k in sorted(wcss)}
silhouette_by_k_json = json.dumps({str(k): float(silhouettes[k]) for k in sorted(silhouettes)})
wcss_by_k_json = json.dumps(elbow_data)

scaler_data = {
    "mean": scaler.mean_.tolist(),
    "scale": scaler.scale_.tolist(),
    "features": CLUSTERING_FEATURES,
}

segment_models_meta = {
    "model_version": "kmeans_v1",
    "k": chosen_k,
    "centroids": centroids_original.tolist(),
    "scaler": scaler_data,
    "wcss": float(final_km.inertia_),
    "silhouette": float(final_silhouette) if final_silhouette else None,
    "elbow_data": elbow_data,
    "wcss_by_k": wcss_by_k_json,
    "silhouette_by_k": silhouette_by_k_json,
}
with open(MODEL_DIR / "segment_models_meta.json", "w") as f:
    json.dump(segment_models_meta, f, indent=2)

# Save driver_segments assignments
driver_segments = []
for idx, (_, driver_row) in enumerate(feature_df.iterrows()):
    seg_id = int(labels[idx])
    seg_label = centroids_df.iloc[seg_id]["segment_label"]
    distance = float(min_distance[idx])
    atypical = bool(is_atypical[idx])
    driver_segments.append({
        "driver_profile_id": int(driver_row["driver_profile_id"]),
        "segment_id": seg_id,
        "segment_label": seg_label,
        "distance_to_centroid": distance,
        "model_version": "kmeans_v1",
        "is_atypical": atypical,
        "pca_x": float(pca_coords[idx][0]),
        "pca_y": float(pca_coords[idx][1]),
    })

with open(MODEL_DIR / "driver_segments.json", "w") as f:
    json.dump(driver_segments, f, indent=2)

print(f"\nPersisted {len(driver_segments)} driver segment assignments")
print("Model artifacts saved to", MODEL_DIR)

display(Image(filename=str(EDA_DIR / "elbow_plot.png")))
display(Image(filename=str(EDA_DIR / "silhouette_plot.png")))

## Step 5: Association Rule Mining

Builds per-violation transaction rows, then:
- Runs FP-Growth (and compares with Apriori)
- Sweeps min_support 0.03-0.10
- Verifies the lecture example: Cooking oil -> Beef = Support 14.3%, Confidence 66.7%, Coverage 21.4%, Lift 1.556
- Mines final rules at min_support=0.05, min_confidence=0.60
- Computes Support, Confidence, Coverage, Lift for each rule
- Ranks by transparent composite score

Never pre-clusters before mining (avoids hiding cross-category rules).

In [ ]:
# Build per-violation transactions (one-hot encoded basket)
transactions_rows = []
for i, row in df.iterrows():
    vid = i + 1
    offense = row["offense_type"]
    zone = row["zone_type"]
    weather = row["weather_conditions"]
    
    # Derive time-of-day from index (deterministic)
    hour = (i * 7) % 24
    is_night = hour >= 18 or hour <= 6
    
    # Payment behavior correlated with driver risk
    driver_risk = row["reoffended_within_180_days"]
    payment_late = bool(rng.random() < (0.3 + 0.4 * driver_risk))
    payment_completed = bool(rng.random() < (0.7 - 0.3 * driver_risk))
    payment_pending = not (payment_late or payment_completed)
    
    appeal_filed = bool(rng.random() < 0.15)
    reoffended_180d = int(row["reoffended_within_180_days"])
    
    transactions_rows.append({
        "violation_id": vid,
        "driver_profile_id": (i % 400) + 1,
        "type_speeding": offense == "speeding",
        "type_red_light": offense == "red_light",
        "type_dui": offense == "dui",
        "type_no_seatbelt": offense == "no_seatbelt",
        "type_phone_use": offense == "phone_use",
        "type_reckless_driving": offense == "reckless_driving",
        "type_illegal_parking": offense == "illegal_parking",
        "type_driving_without_license": offense == "driving_without_license",
        "type_overloading": offense == "overloading",
        "type_no_insurance": offense == "no_insurance",
        "zone_urban": zone == "urban",
        "zone_highway": zone == "highway",
        "zone_school_zone": zone == "school_zone",
        "zone_residential": zone == "residential",
        "weather_clear": weather == "clear",
        "weather_rain": weather == "rain",
        "weather_fog": weather == "fog",
        "weather_night": weather == "night",
        "time_night": is_night,
        "time_day": not is_night,
        "payment_late": payment_late,
        "payment_completed": payment_completed,
        "payment_pending": payment_pending,
        "appeal_filed": appeal_filed,
        "outcome_reoffended_180d": reoffended_180d,
        "is_synthetic": 1,
    })

transactions_df = pd.DataFrame(transactions_rows)
print(f"Transactions: {len(transactions_df)} rows")
print(f"Columns: {list(transactions_df.columns)}")

ITEM_COLUMNS = [
    "type_speeding", "type_red_light", "type_dui", "type_no_seatbelt",
    "type_phone_use", "type_reckless_driving", "type_illegal_parking",
    "type_driving_without_license", "type_overloading", "type_no_insurance",
    "zone_urban", "zone_highway", "zone_school_zone", "zone_residential",
    "weather_clear", "weather_rain", "weather_fog", "weather_night",
    "time_night", "time_day",
    "payment_late", "payment_completed", "payment_pending",
    "appeal_filed", "outcome_reoffended_180d",
]

RISK_CONSEQUENTS = {"outcome_reoffended_180d", "payment_late"}
MODEL_VERSION = "rules_v1"

def item_name(col):
    mapping = {
        "type_speeding": "type=speeding",
        "type_red_light": "type=red_light",
        "type_dui": "type=dui",
        "type_no_seatbelt": "type=no_seatbelt",
        "type_phone_use": "type=phone_use",
        "type_reckless_driving": "type=reckless_driving",
        "type_illegal_parking": "type=illegal_parking",
        "type_driving_without_license": "type=driving_without_license",
        "type_overloading": "type=overloading",
        "type_no_insurance": "type=no_insurance",
        "zone_urban": "zone=urban",
        "zone_highway": "zone=highway",
        "zone_school_zone": "zone=school_zone",
        "zone_residential": "zone=residential",
        "weather_clear": "weather=clear",
        "weather_rain": "weather=rain",
        "weather_fog": "weather=fog",
        "weather_night": "weather=night",
        "time_night": "time=night",
        "time_day": "time=day",
        "payment_late": "payment=late",
        "payment_completed": "payment=completed",
        "payment_pending": "payment=pending",
        "appeal_filed": "appeal=filed",
        "outcome_reoffended_180d": "outcome=reoffended_180d",
    }
    return mapping.get(col, col)

def compute_measures(basket, antecedent, consequent, n):
    a_mask = basket[list(antecedent)].all(axis=1)
    b_mask = basket[list(consequent)].all(axis=1)
    ab_mask = a_mask & b_mask
    support_ab = ab_mask.sum() / n
    support_a = a_mask.sum() / n
    support_b = b_mask.sum() / n
    confidence = support_ab / support_a if support_a > 0 else 0
    coverage = support_a
    lift = support_ab / (support_a * support_b) if (support_a * support_b) > 0 else 0
    return {
        "support": round(float(support_ab), 6),
        "confidence": round(float(confidence), 6),
        "coverage": round(float(coverage), 6),
        "lift": round(float(lift), 6),
    }

def composite_score(support, confidence, coverage, lift):
    s = min(support * 10, 1.0)
    c = confidence
    cov = min(coverage * 5, 1.0)
    l = min(lift / 3.0, 1.0)
    return (s * c * cov * l) ** 0.25

def plain_english(antecedent_items, consequent_items, support, confidence, coverage, lift):
    a_str = ", ".join(antecedent_items)
    b_str = ", ".join(consequent_items)
    return (
        f"When {a_str} occurs, {b_str} also occurs in "
        f"{confidence*100:.1f}% of cases, "
        f"{lift:.2f}x more often than chance "
        f"(covers {coverage*100:.1f}% of all violations)."
    )

# --- Lecture example verification ---
print("\n" + "="*60)
print("Lecture Example Unit Test")
print("="*60)
lecture_data = [
    {"Cooking oil": 1, "Beef": 1, "Milk": 0},
    {"Cooking oil": 1, "Beef": 1, "Milk": 0},
    {"Cooking oil": 1, "Beef": 0, "Milk": 0},
    {"Cooking oil": 0, "Beef": 1, "Milk": 0},
    {"Cooking oil": 0, "Beef": 1, "Milk": 0},
    {"Cooking oil": 0, "Beef": 1, "Milk": 0},
    {"Cooking oil": 0, "Beef": 1, "Milk": 0},
    {"Cooking oil": 0, "Beef": 0, "Milk": 0},
    {"Cooking oil": 0, "Beef": 0, "Milk": 0},
    {"Cooking oil": 0, "Beef": 0, "Milk": 0},
    {"Cooking oil": 0, "Beef": 0, "Milk": 0},
    {"Cooking oil": 0, "Beef": 0, "Milk": 0},
    {"Cooking oil": 0, "Beef": 0, "Milk": 0},
    {"Cooking oil": 0, "Beef": 0, "Milk": 0},
]
lecture_basket = pd.DataFrame(lecture_data)
n_lecture = len(lecture_basket)
A = frozenset(["Cooking oil"])
B = frozenset(["Beef"])
m = compute_measures(lecture_basket, A, B, n_lecture)
print(f"  Support:  {m['support']*100:.1f}%  (expected 14.3%)")
print(f"  Confidence: {m['confidence']*100:.1f}%  (expected 66.7%)")
print(f"  Coverage:  {m['coverage']*100:.1f}%  (expected 21.4%)")
print(f"  Lift:      {m['lift']:.3f}  (expected 1.556)")
assert abs(m["support"] - 0.143) < 0.001, f"Support {m['support']} != 0.143"
assert abs(m["confidence"] - 0.667) < 0.001, f"Confidence {m['confidence']} != 0.667"
assert abs(m["coverage"] - 0.214) < 0.001, f"Coverage {m['coverage']} != 0.214"
assert abs(m["lift"] - 1.556) < 0.001, f"Lift {m['lift']} != 1.556"
print("  PASSED!")

# --- Sweep min_support ---
print("\nSweeping min_support 0.03-0.10 (FP-Growth)...")
support_levels = [0.03, 0.04, 0.05, 0.06, 0.07, 0.08, 0.09, 0.10]
for ms in support_levels:
    itemsets = fpgrowth(transactions_df[ITEM_COLUMNS].astype(int), min_support=ms, use_colnames=True, max_len=None)
    if len(itemsets) > 0:
        rules = mlxtend_rules(itemsets, metric="confidence", min_threshold=0.60)
        rules = rules[rules["antecedents"].apply(len) <= 3]
        n_rules = len(rules)
    else:
        n_rules = 0
    print(f"  min_support={ms}: {len(itemsets)} itemsets, {n_rules} rules (conf>=0.60)")

# --- Mine final rules ---
min_support = 0.05
min_confidence = 0.60
print(f"\nMining rules: min_support={min_support}, min_confidence={min_confidence}")
basket = transactions_df[ITEM_COLUMNS].astype(int)
n = len(basket)
itemsets = fpgrowth(basket, min_support=min_support, use_colnames=True)
print(f"  Found {len(itemsets)} frequent itemsets")
rules = mlxtend_rules(itemsets, metric="confidence", min_threshold=min_confidence)
print(f"  Found {len(rules)} rules before antecedent filter")
rules = rules[rules["antecedents"].apply(len) <= 3]
print(f"  {len(rules)} rules after antecedent length <= 3 filter")

if len(rules) == 0:
    print("  No rules found! Try lowering min_support.")
else:
    rule_records = []
    for _, rule in rules.iterrows():
        antecedent = frozenset(rule["antecedents"])
        consequent = frozenset(rule["consequents"])
        m = compute_measures(basket, antecedent, consequent, n)
        antecedent_items = sorted([item_name(c) for c in antecedent])
        consequent_items = sorted([item_name(c) for c in consequent])
        is_risk_rule = any(c in RISK_CONSEQUENTS for c in consequent_items)
        rule_family = "risk" if is_risk_rule else "context"
        if m["lift"] > 1.05:
            association = "positive"
        elif m["lift"] < 0.95:
            association = "negative"
        else:
            association = "independence"
        comp = composite_score(m["support"], m["confidence"], m["coverage"], m["lift"])
        rule_records.append({
            "antecedent": antecedent_items,
            "consequent": consequent_items,
            "support": m["support"],
            "confidence": m["confidence"],
            "coverage": m["coverage"],
            "lift": m["lift"],
            "rule_family": rule_family,
            "association": association,
            "composite_score": round(float(comp), 6),
            "plain_english": plain_english(
                antecedent_items, consequent_items,
                m["support"], m["confidence"], m["coverage"], m["lift"]
            ),
            "min_support": min_support,
            "min_confidence": min_confidence,
        })

    rule_records.sort(key=lambda r: r["composite_score"], reverse=True)

    print("\nTop 10 rules by composite score:")
    for i, r in enumerate(rule_records[:10]):
        print(f"  {i+1}. {r['plain_english'][:100]}... (composite={r['composite_score']:.3f})")

    # Save rules artifact
    rules_artifact = {
        "rules": rule_records,
        "item_columns": ITEM_COLUMNS,
        "n_transactions": n,
    }
    joblib.dump(rules_artifact, MODEL_DIR / "association_rules.joblib")
    print(f"\nRules artifact saved to {MODEL_DIR / 'association_rules.joblib'}")
    print(f"Total rules persisted: {len(rule_records)}")

print("\nAssociation rule mining complete!")

## Step 6: Supervised Risk Model

Builds training data from per-driver features + segment assignments + rule hits.

### Ablation Study
(A) original 6 features
(B) A + segmentation
(C) A + rule-hit features
(D) A + both

Models: RandomForest, GradientBoosting, LogisticRegression
5-fold stratified CV with calibration

### Final Model
CalibratedClassifierCV (isotonic) on best feature set + model

### Fairness Report
FPR by zone_type, weather_conditions, segment_id; flag groups > 1.25x overall FPR

In [1]:
# Build training dataset: one row per driver's latest violation
# Features: original 6 + violation_count + new_driver + segment_id + distance_to_centroid + rule_hit_count

# Load segments
with open(MODEL_DIR / "driver_segments.json") as f:
    driver_segments_list = json.load(f)
seg_df = pd.DataFrame(driver_segments_list)

# Merge segment info onto feature_df
df_train = feature_df.merge(
    seg_df[['driver_profile_id', 'segment_id', 'segment_label', 'distance_to_centroid']],
    on='driver_profile_id',
    how='left',
)
df_train['segment_id'] = df_train['segment_id'].fillna(-1).astype(int)
df_train['distance_to_centroid'] = df_train['distance_to_centroid'].fillna(999.0)

# Compute rule_hit_count: how many risk rule antecedents match this violation
# Use zone_type and weather_conditions from the original df
zone_map = dict(zip(df.index, df['zone_type']))
weather_map = dict(zip(df.index, df['weather_conditions']))

df_train['rule_hit_count'] = 0
if len(rule_records) > 0:
    for i, row in df_train.iterrows():
        hits = 0
        for rule in rule_records:
            match = True
            for item in rule['antecedent']:
                if item.startswith('zone='):
                    zone_val = item.split('=', 1)[1]
                    if str(row.get('zone_type', zone_map.get(i, 'urban'))) != zone_val:
                        match = False
                        break
                elif item.startswith('weather='):
                    weather_val = item.split('=', 1)[1]
                    if str(row.get('weather_conditions', weather_map.get(i, 'clear'))) != weather_val:
                        match = False
                        break
                else:
                    match = False
                    break
            if match:
                hits += 1
        df_train.at[i, 'rule_hit_count'] = hits

print(f"\nTraining records: {len(df_train)}")
print(f"Target distribution: {df_train['reoffended_within_180_days'].value_counts().to_dict()}")
display(df_train.head(5))

# --- Ablation study ---
ORIGINAL_6 = [
    "days_since_last_offense",
    "speed_recorded",
    "zone_type",
    "weather_conditions",
    "cumulative_active_demerit_points",
    "past_fine_payment_delays_days",
]
SEGMENT_FEATURES = ["violation_count", "new_driver", "distance_to_centroid"]
RULE_FEATURE = "rule_hit_count"

feature_sets = {
    "A_original_6": ORIGINAL_6,
    "B_plus_segmentation": ORIGINAL_6 + SEGMENT_FEATURES + ["segment_id"],
    "C_plus_rules": ORIGINAL_6 + [RULE_FEATURE],
    "D_plus_both": ORIGINAL_6 + SEGMENT_FEATURES + ["segment_id", RULE_FEATURE],
}

models_to_test = ["random_forest", "gradient_boosting", "logistic_regression"]
cv = StratifiedKFold(n_splits=5, shuffle=True, random_state=RANDOM_STATE)

def build_pipeline(feature_cols, numeric_cols, categorical_cols, model_name):
    preprocess = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline([("imputer", SimpleImputer(strategy="median"))]),
                numeric_cols,
            ),
            (
                "cat",
                Pipeline([
                    ("imputer", SimpleImputer(strategy="most_frequent")),
                    ("onehot", OneHotEncoder(handle_unknown="ignore")),
                ]),
                categorical_cols,
            ),
        ]
    )
    if model_name == "random_forest":
        model = RandomForestClassifier(
            n_estimators=300, max_depth=10, min_samples_leaf=3,
            random_state=RANDOM_STATE, n_jobs=-1, class_weight="balanced",
        )
    elif model_name == "gradient_boosting":
        model = GradientBoostingClassifier(
            n_estimators=200, max_depth=5, random_state=RANDOM_STATE,
        )
    elif model_name == "logistic_regression":
        model = LogisticRegression(
            max_iter=2000, random_state=RANDOM_STATE, class_weight="balanced",
        )
    else:
        raise ValueError(f"Unknown model: {model_name}")
    return Pipeline([("preprocess", preprocess), ("clf", model)])

ablation_results = []

for fs_name, fs_cols in feature_sets.items():
    print(f"\n--- Feature Set: {fs_name} ---")
    avail_cols = [c for c in fs_cols if c in df_train.columns]
    missing = [c for c in fs_cols if c not in df_train.columns]
    if missing:
        print(f"  Missing columns: {missing}")
    
    X = df_train[avail_cols].copy()
    y = df_train[TARGET_COLUMN].astype(int)
    
    numeric_cols = [c for c in avail_cols if c not in ["zone_type", "weather_conditions", "segment_id"]]
    categorical_cols = [c for c in avail_cols if c in ["zone_type", "weather_conditions", "segment_id"]]
    
    for model_name in models_to_test:
        fold_scores = {"roc_auc": [], "f1": [], "precision": [], "recall": []}
        
        for train_idx, test_idx in cv.split(X, y):
            X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
            y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
            
            pipe = build_pipeline(avail_cols, numeric_cols, categorical_cols, model_name)
            pipe.fit(X_train, y_train)
            
            y_proba = pipe.predict_proba(X_test)[:, 1]
            y_pred = (y_proba >= 0.5).astype(int)
            
            fold_scores["roc_auc"].append(roc_auc_score(y_test, y_proba))
            fold_scores["f1"].append(f1_score(y_test, y_pred, zero_division=0))
            fold_scores["precision"].append(precision_score(y_test, y_pred, zero_division=0))
            fold_scores["recall"].append(recall_score(y_test, y_pred, zero_division=0))
        
        mean_auc = np.mean(fold_scores["roc_auc"])
        std_auc = np.std(fold_scores["roc_auc"])
        mean_f1 = np.mean(fold_scores["f1"])
        
        ablation_results.append({
            "feature_set": fs_name,
            "model": model_name,
            "roc_auc_mean": round(float(mean_auc), 4),
            "roc_auc_std": round(float(std_auc), 4),
            "f1_mean": round(float(mean_f1), 4),
            "precision_mean": round(float(np.mean(fold_scores["precision"])), 4),
            "recall_mean": round(float(np.mean(fold_scores["recall"])), 4),
        })
        print(f"  {model_name}: ROC-AUC={mean_auc:.4f} (+/-{std_auc:.4f}), F1={mean_f1:.4f}")

ablation_df = pd.DataFrame(ablation_results)
ablation_df.to_json(REPORTS_DIR / "ablation_report.json", orient="records", indent=2)
print(f"\nAblation report saved to {REPORTS_DIR / 'ablation_report.json'}")

best_row = ablation_df.loc[ablation_df["roc_auc_mean"].idxmax()]
best_fs = best_row["feature_set"]
best_model = best_row["model"]
print(f"\nBest: {best_fs} + {best_model} (ROC-AUC={best_row['roc_auc_mean']:.4f})")

fs_map = {
    "A_original_6": ORIGINAL_6,
    "B_plus_segmentation": ORIGINAL_6 + SEGMENT_FEATURES + ["segment_id"],
    "C_plus_rules": ORIGINAL_6 + [RULE_FEATURE],
    "D_plus_both": ORIGINAL_6 + SEGMENT_FEATURES + ["segment_id", RULE_FEATURE],
}
best_features = fs_map.get(best_fs, ORIGINAL_6)

# --- Final model training ---
print("\n" + "="*60)
print("Final Model Training")
print("="*60)

avail_cols = [c for c in best_features if c in df_train.columns]
numeric_cols = [c for c in avail_cols if c not in ["zone_type", "weather_conditions", "segment_id"]]
categorical_cols = [c for c in avail_cols if c in ["zone_type", "weather_conditions", "segment_id"]]

X = df_train[avail_cols]
y = df_train[TARGET_COLUMN].astype(int)

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=RANDOM_STATE, stratify=y
)

pipe = build_pipeline(avail_cols, numeric_cols, categorical_cols, best_model)
calibrated = CalibratedClassifierCV(pipe, method="isotonic", cv=3)
calibrated.fit(X_train, y_train)

y_proba = calibrated.predict_proba(X_test)[:, 1]
y_pred = (y_proba >= 0.5).astype(int)

auc = roc_auc_score(y_test, y_proba)
f1 = f1_score(y_test, y_pred, zero_division=0)
precision = precision_score(y_test, y_pred, zero_division=0)
recall = recall_score(y_test, y_pred, zero_division=0)

print(f"  Test ROC-AUC: {auc:.4f}")
print(f"  Test F1: {f1:.4f}")
print(f"  Test Precision: {precision:.4f}")
print(f"  Test Recall: {recall:.4f}")
print(f"\nClassification Report:")
print(classification_report(y_test, y_pred, digits=4))

# Save model artifact
model_version = "rf_v2"
risk_artifact = {
    "model": calibrated,
    "feature_cols": avail_cols,
    "numeric_cols": numeric_cols,
    "categorical_cols": categorical_cols,
    "model_version": model_version,
}
joblib.dump(risk_artifact, MODEL_DIR / "risk_model_rf_v2.joblib")
print(f"\nModel saved to {MODEL_DIR / 'risk_model_rf_v2.joblib'}")

# --- Fairness report ---
print("\n" + "="*60)
print("Fairness Report")
print("="*60)

df_train["y_pred"] = y_pred
df_train["y_true"] = y

overall_fpr = ((df_train["y_pred"] == 1) & (df_train["y_true"] == 0)).sum() / max((df_train["y_true"] == 0).sum(), 1)

fairness = {"overall_fpr": round(float(overall_fpr), 4)}
print(f"Overall FPR: {overall_fpr:.4f}")

for group_col in ["zone_type", "weather_conditions", "segment_id"]:
    if group_col not in df_train.columns:
        continue
    group_fairness = {}
    for val in df_train[group_col].unique():
        subset = df_train[df_train[group_col] == val]
        if len(subset) == 0:
            continue
        fpr = ((subset["y_pred"] == 1) & (subset["y_true"] == 0)).sum() / max((subset["y_true"] == 0).sum(), 1)
        tpr = ((subset["y_pred"] == 1) & (subset["y_true"] == 1)).sum() / max((subset["y_true"] == 1).sum(), 1)
        flagged = bool(fpr > overall_fpr * 1.25)
        group_fairness[str(val)] = {
            "fpr": round(float(fpr), 4),
            "tpr": round(float(tpr), 4),
            "n": int(len(subset)),
            "fpr_ratio_to_overall": round(float(fpr / max(overall_fpr, 0.001)), 4),
            "flagged": flagged,
        }
    fairness[group_col] = group_fairness
    flagged_groups = [k for k, v in group_fairness.items() if v["flagged"]]
    if flagged_groups:
        print(f"  {group_col}: FLAGGED groups: {flagged_groups}")
    else:
        print(f"  {group_col}: No groups flagged")

with open(REPORTS_DIR / "fairness_report.json", "w") as f:
    json.dump(fairness, f, indent=2)
print(f"\nFairness report saved to {REPORTS_DIR / 'fairness_report.json'}")

print("\nSupervised model training complete!")

NameError: name 'MODEL_DIR' is not defined

## Step 7: Download Artifacts

All trained model artifacts and reports are in `/mnt/data/traffic_demerit_ml/`.

Copy these files into your Laravel ML service:
- `model/risk_model_rf_v2.joblib` → `ml_service/model/risk_model_rf_v2.joblib`
- `model/kmeans_model.joblib` → `ml_service/model/kmeans_model.joblib`
- `model/kmeans_scaler.joblib` → `ml_service/model/kmeans_scaler.joblib`
- `model/association_rules.joblib` → `ml_service/model/association_rules.joblib`
- `reports/ablation_report.json` → `ml_service/reports/ablation_report.json`
- `reports/fairness_report.json` → `ml_service/reports/fairness_report.json`
- `reports/eda/*.png` → `ml_service/reports/eda/*.png`

Then restart the FastAPI service: `uvicorn app:app --host 127.0.0.1 --port 8001 --reload`

In [ ]:
import zipfile
from pathlib import Path

zip_path = OUTPUT_DIR.parent / "traffic_demerit_ml_artifacts.zip"

with zipfile.ZipFile(zip_path, "w", zipfile.ZIP_DEFLATED) as zf:
    for p in MODEL_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(OUTPUT_DIR.parent))
    for p in REPORTS_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(OUTPUT_DIR.parent))
    for p in DATA_DIR.rglob("*"):
        if p.is_file():
            zf.write(p, p.relative_to(OUTPUT_DIR.parent))

print(f"Artifacts zipped to: {zip_path}")
print(f"Zip size: {zip_path.stat().st_size / 1024 / 1024:.2f} MB")

display(FileLink(str(zip_path)))

## Summary

Training complete. Key outputs:
- **Dataset**: `/mnt/data/traffic_demerit_ml/data/simulated_reoffending.csv` (2,500 rows, synthetic)
- **Risk model**: `model/risk_model_rf_v2.joblib` (calibrated RandomForest, rf_v2)
- **Segmentation**: `model/kmeans_model.joblib` + `model/kmeans_scaler.joblib` (k-means_v1)
- **Association rules**: `model/association_rules.joblib` (rules_v1)
- **Ablation report**: `reports/ablation_report.json`
- **Fairness report**: `reports/fairness_report.json`
- **EDA plots**: `reports/eda/` (elbow, silhouette, distributions, outliers, correlation heatmap)

All data is synthetic and for demonstration purposes only.